# 03 — Score one run with the frozen instrument

Run this **in the same session as notebook 02**, or point `RUN` at an earlier training
notebook's output.

Reports the primary metric and, beside it, the lexical anchor **recomputed on these same
generated captions at this same n**. §4 forbids quoting one without the other — three
findings were already retracted for comparing an anchor at one n against an accuracy at
another. The number the study actually claims is the **margin**, not the accuracy.

The classifier is loaded and never trained; its sha256 goes into the output so a result can
be traced to the instrument that produced it.

**Attach both datasets:** `emocap-v2-arms` and `emocap-v2-classifier`.

Scoring can also be done at home instead, which needs no classifier upload at all —
predictions are 278 bytes per cell:

    uv run python scripts/pull_kaggle_runs.py --kernel <owner>/<training-notebook-slug>

In [ ]:
# ── parameters ────────────────────────────────────────────────────────────
RUN = "/kaggle/working/runs/S_paired5-f0"   # a directory holding predictions.jsonl

# TWO datasets, attach BOTH for this notebook. The single 421 MB upload stalled for 54
# minutes and created nothing, so the parts are split by when they are needed: `arms` and
# `features` to TRAIN, the classifier to SCORE.
DATA = "/kaggle/input/emocap-v2-arms"
CLASSIFIER = "/kaggle/input/emocap-v2-classifier/classifier"

In [ ]:
import json, subprocess, sys
from pathlib import Path
sys.path.insert(0, "/kaggle/working/EmoCap/src")

# Same script that runs locally -- no notebook-only analysis path.
!{sys.executable} /kaggle/working/EmoCap/scripts/score_arm.py \
    --run {RUN} --classifier {CLASSIFIER}

In [ ]:
score = json.loads(Path(RUN, "score.json").read_text())

acc, anc, ch = score["accuracy"], score["anchor"], score["chance"]
lo, hi = score["ci"]["lo"], score["ci"]["hi"]
print(f"arm {score['arm']}  fold {score['fold']}  "
      f"{score['n_cells']:,} cells / {score['n_images']:,} images")
print(f"  accuracy  {acc:.4f}  [{lo:.4f}, {hi:.4f}]")
print(f"  anchor    {anc:.4f}  (same captions, n={score['anchor_n_cells']:,})")
print(f"  margin    {acc - anc:+.4f}")
print(f"  chance    {ch:.4f}")

# The two questions this run exists to answer, stated before the sweep is funded.
above_chance = lo > ch
above_anchor = acc > anc
print(f"\n  CI clears chance?  {above_chance}")
print(f"  beats own anchor?  {above_anchor}")
if not above_chance:
    print("\n  This arm did not learn the register. A comparison against another arm that")
    print("  also sits at floor is null by construction, not by finding.")

In [ ]:
# Per-register recall. Two registers collapsing into each other is a finding about the
# taxonomy (§4.3), not noise -- and a single register absorbing everything means the
# conditioning is being ignored rather than used.
rec = score["confusion"]["recall"]
for k, v in rec.items():
    bar = "#" * int(round(v * 40))
    print(f"  {k:<9} {v:.3f} {bar}")
print(f"\n  spread {max(rec.values()) - min(rec.values()):.3f}")
print(f"  captions: {score['unique_captions']:,} unique of {score['n_cells']:,}, "
      f"{score['mean_words']} mean words, {score['empty_captions']} empty")

In [ ]:
# Row-normalised confusion, for the write-up.
cm = score["confusion"]["row_normalised"]
regs = list(cm)
print("true \\ pred   " + "".join(f"{r[:4]:>7}" for r in regs))
for t in regs:
    print(f"{t:<13} " + "".join(f"{cm[t][p]:>7.3f}" for p in regs))

In [ ]:
# Eyeball the captions. Numbers hide a model that emits fluent text with no register.
preds = [json.loads(l) for l in Path(RUN, "predictions.jsonl").read_text().splitlines()]
by_img = {}
for p in preds:
    by_img.setdefault(p["image_id"], []).append(p)

for img, cells in list(by_img.items())[:3]:
    if len(cells) < 2:
        continue
    print(img)
    for c in sorted(cells, key=lambda c: c["emotion"]):
        print(f"  [{c['emotion']:<9}] {c['generated']}")
    print()